# StyleGAN-NADA reimplementation, DLS final project

## Cloning a repository

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content
!git clone https://github.com/Chinozor/clip_nada_reimplementation_project.git project
!ls -lah /content/project

/content
Cloning into 'project'...
remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 89 (delta 36), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (89/89), 23.80 KiB | 7.93 MiB/s, done.
Resolving deltas: 100% (36/36), done.
total 52K
drwxr-xr-x 4 root root 4.0K Feb  4 18:13 .
drwxr-xr-x 1 root root 4.0K Feb  4 18:13 ..
drwxr-xr-x 5 root root 4.0K Feb  4 18:13 configs
-rw-r--r-- 1 root root 1.7K Feb  4 18:13 freezing.py
drwxr-xr-x 8 root root 4.0K Feb  4 18:13 .git
-rw-r--r-- 1 root root 4.4K Feb  4 18:13 losses.py
-rw-r--r-- 1 root root   58 Feb  4 18:13 README.md
-rw-r--r-- 1 root root 4.9K Feb  4 18:13 trainers.py
-rw-r--r-- 1 root root 4.3K Feb  4 18:13 train.py
-rw-r--r-- 1 root root 1.2K Feb  4 18:13 utils.py


## Installing dependencies

In [3]:
%cd /content/project
!pip -q install -r /content/project/configs/requirements.txt

/content/project
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 129.9 MB/s eta 0:00:00


## Downloading StyleGAN-2, weights

In [4]:
%cd /content/project
!mkdir -p third_party
!test -d third_party/stylegan2-pytorch || git clone https://github.com/rosinality/stylegan2-pytorch.git third_party/stylegan2-pytorch

/content/project
Cloning into 'third_party/stylegan2-pytorch'...
remote: Enumerating objects: 395, done.
remote: Total 395 (delta 0), reused 0 (delta 0), pack-reused 395 (from 1)
Receiving objects: 100% (395/395), 122.51 MiB | 11.94 MiB/s, done.
Resolving deltas: 100% (205/205), done.


In [5]:
%cd /content/project
!python configs/scripts/download_weights.py
!ls -lh weights

/content/project
[download] stylegan2-ffhq-config-f.pt
Downloading...
From (original): https://drive.google.com/uc?id=1EM87UquaoQmk17Q8d5kYIAHqu0dkYqdT
From (redirected): https://drive.google.com/uc?id=1EM87UquaoQmk17Q8d5kYIAHqu0dkYqdT&confirm=t&uuid=6da48231-5b95-4352-8866-a380d02acc08
To: /content/project/weights/stylegan2-ffhq-config-f.pt
100% 381M/381M [00:08<00:00, 44.8MB/s]
[download] model_ir_se50.pth
Downloading...
From (original): https://drive.google.com/uc?id=1N0MZSqPRJpLfP4mFQCS14ikrVSe8vQlL
From (redirected): https://drive.google.com/uc?id=1N0MZSqPRJpLfP4mFQCS14ikrVSe8vQlL&confirm=t&uuid=cb6c52bf-8d3d-4f85-a9ba-acc95b59eef7
To: /content/project/weights/model_ir_se50.pth
100% 175M/175M [00:03<00:00, 55.2MB/s]
[ok] weights: ['model_ir_se50.pth', 'stylegan2-ffhq-config-f.pt']
total 532M
-rw-r--r-- 1 root root 168M Aug 14  2021 model_ir_se50.pth
-rw-r--r-- 1 root root 364M Sep 10  2020 stylegan2-ffhq-config-f.pt


In [6]:
%%bash
mv /content/project/third_party/stylegan2-pytorch /content/project/third_party/stylegan2_pytorch

In [7]:
%%bash
set -euo pipefail

FILE="/content/project/third_party/stylegan2_pytorch/model.py"

python - <<'PY'
from pathlib import Path
p = Path("/content/project/third_party/stylegan2_pytorch/model.py")
t = p.read_text()

old = "from op import FusedLeakyReLU, fused_leaky_relu, upfirdn2d, conv2d_gradfix"
new = "from .op import FusedLeakyReLU, fused_leaky_relu, upfirdn2d, conv2d_gradfix"

if old not in t:
    raise SystemExit("Не нашёл ожидаемую строку импорта в model.py — проверь файл вручную.")
p.write_text(t.replace(old, new))
print("OK: patched", p)
PY

OK: patched /content/project/third_party/stylegan2_pytorch/model.py


## Bringing prompt directories into the required format

In [8]:
%%bash
set -euo pipefail

mkdir -p /content/project/configs/domains

cat > /content/project/configs/domains/anime.yaml <<'YAML'
name: tgt_anime_face
prompts:
  src: configs/prompts/src_face.yaml
  tgt: configs/prompts/tgt_anime_face.yaml
  global: configs/prompts/tgt_anime_face.yaml
YAML

cat > /content/project/configs/domains/statue.yaml <<'YAML'
name: tgt_statue_face
prompts:
  src: configs/prompts/src_face.yaml
  tgt: configs/prompts/tgt_statue_face.yaml
  global: configs/prompts/tgt_statue_face.yaml
YAML

cat > /content/project/configs/domains/werewolf.yaml <<'YAML'
name: tgt_werewolf_face
prompts:
  src: configs/prompts/src_face.yaml
  tgt: configs/prompts/tgt_werewolf_face.yaml
  global: configs/prompts/tgt_werewolf_face.yaml
YAML

cat > /content/project/configs/domains/white_walker.yaml <<'YAML'
name: tgt_white_walker_face
prompts:
  src: configs/prompts/src_face.yaml
  tgt: configs/prompts/tgt_white_walker_face.yaml
  global: configs/prompts/tgt_white_walker_face.yaml
YAML

echo "OK"

OK


## Train model with differents prompts and hyperparams

In [ ]:
%%bash
set -euo pipefail

OUT_ROOT="/content/drive/MyDrive/clip_gan_outputs"
mkdir -p "$OUT_ROOT"

cd /content/project
export PYTHONPATH="/content:${PYTHONPATH:-}"

test -f weights/stylegan2-ffhq-config-f.pt
test -f train.py

DOMAINS=(
  "configs/domains/anime.yaml"
  "configs/domains/statue.yaml"
  "configs/domains/werewolf.yaml"
  "configs/domains/white_walker.yaml"
)

HYPERS=(
  "configs/exp/base.yaml"
  "configs/exp/topk_6.yaml"
  "configs/exp/topk_8.yaml"
  "configs/exp/topk_18.yaml"
  "configs/exp/policy_full.yaml"
  "configs/exp/no_selection.yaml"
)

BASE="configs/exp/base.yaml"

echo "OUT_ROOT=$OUT_ROOT"
echo "BASE=$BASE"

for D in "${DOMAINS[@]}"; do
  for H in "${HYPERS[@]}"; do
    TAG="$(basename "$D" .yaml)__$(basename "$H" .yaml)"
    echo "RUN $TAG"

    python -u train.py \
      --base "$BASE" \
      --domain "$D" \
      --hyper "$H" \
      --outputs_root "$OUT_ROOT" \
      2>&1 | tee -a "$OUT_ROOT/${TAG}.log"
  done
done


OUT_ROOT=/content/drive/MyDrive/clip_gan_outputs
BASE=configs/exp/base.yaml
RUN anime__base
100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 123MiB/s]
/content/project/third_party/stylegan2_pytorch/op/conv2d_gradfix.py:88: UserWarning: conv2d_gradfix not supported on PyTorch 2.9.0+cu126. Falling back to torch.nn.functional.conv2d().
  warnings.warn(
0 1.0
Figure(600x600)
/content/project/third_party/stylegan2_pytorch/op/conv2d_gradfix.py:88: UserWarning: conv2d_gradfix not supported on PyTorch 2.9.0+cu126. Falling back to torch.nn.functional.conv2d().
  warnings.warn(
100 0.8401423692703247
Figure(600x600)
200 0.819865345954895
Figure(600x600)
300 0.7931523323059082
Figure(600x600)
400 0.7719613313674927
Figure(600x600)
500 0.7838698625564575
Figure(600x600)
600 0.7734434604644775
Figure(600x600)
700 0.7601749897003174
Figure(600x600)
800 0.7468425035476685
Figure(600x600)
900 0.746033787727356
Figure(600x600)
RUN anime__topk_6
/content/project/third_party/stylega

## Train model with best hyperparams (as i think)

In [9]:
%%bash
set -euo pipefail

OUT_ROOT="/content/drive/MyDrive/clip_gan_outputs/clip_gan_outputs_ideal"
mkdir -p "$OUT_ROOT"

cd /content/project
export PYTHONPATH="/content:${PYTHONPATH:-}"

test -f weights/stylegan2-ffhq-config-f.pt
test -f train.py

mkdir -p configs/exp/ideal

cat > configs/exp/ideal/anime_ideal.yaml <<'YAML'
train:
  lr: 0.0015
  batch_size: 4
  steps: 1200
  top_k: 8
  reselection_every: 1
  use_layer_selection: true
  unfreeze_layers_policy: full
YAML

cat > configs/exp/ideal/statue_ideal.yaml <<'YAML'
train:
  lr: 0.0012
  batch_size: 4
  steps: 1400
  top_k: 10
  reselection_every: 1
  use_layer_selection: true
  unfreeze_layers_policy: full
YAML

cat > configs/exp/ideal/werewolf_ideal.yaml <<'YAML'
train:
  lr: 0.0010
  batch_size: 2
  steps: 1800
  top_k: 14
  reselection_every: 1
  use_layer_selection: true
  unfreeze_layers_policy: full
YAML

cat > configs/exp/ideal/white_walker_ideal.yaml <<'YAML'
train:
  lr: 0.0012
  batch_size: 4
  steps: 1500
  top_k: 12
  reselection_every: 1
  use_layer_selection: true
  unfreeze_layers_policy: full
YAML

BASE="configs/exp/base.yaml"

DOMAINS=(
  "configs/domains/anime.yaml"
  "configs/domains/statue.yaml"
  "configs/domains/werewolf.yaml"
  "configs/domains/white_walker.yaml"
)

hyper_for_domain () {
  local d="$1"
  local b
  b="$(basename "$d")"
  case "$b" in
    anime.yaml)        echo "configs/exp/ideal/anime_ideal.yaml" ;;
    statue.yaml)       echo "configs/exp/ideal/statue_ideal.yaml" ;;
    werewolf.yaml)     echo "configs/exp/ideal/werewolf_ideal.yaml" ;;
    white_walker.yaml) echo "configs/exp/ideal/white_walker_ideal.yaml" ;;
    *) echo "" ;;
  esac
}

echo "OUT_ROOT=$OUT_ROOT"
echo "BASE=$BASE"

for D in "${DOMAINS[@]}"; do
  H="$(hyper_for_domain "$D")"
  if [[ -z "$H" ]]; then
    echo "[skip] no ideal hyper for $D"
    continue
  fi

  TAG="$(basename "$D" .yaml)__$(basename "$H" .yaml)"
  echo "RUN $TAG"

  python -u train.py \
    --base "$BASE" \
    --domain "$D" \
    --hyper "$H" \
    --outputs_root "$OUT_ROOT" \
    2>&1 | tee -a "$OUT_ROOT/${TAG}.log"
done


OUT_ROOT=/content/drive/MyDrive/clip_gan_outputs/clip_gan_outputs_ideal
BASE=configs/exp/base.yaml
RUN anime__anime_ideal
100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 347MiB/s]
/content/project/third_party/stylegan2_pytorch/op/conv2d_gradfix.py:88: UserWarning: conv2d_gradfix not supported on PyTorch 2.9.0+cu126. Falling back to torch.nn.functional.conv2d().
  warnings.warn(
0 1.0
Figure(600x600)
/content/project/third_party/stylegan2_pytorch/op/conv2d_gradfix.py:88: UserWarning: conv2d_gradfix not supported on PyTorch 2.9.0+cu126. Falling back to torch.nn.functional.conv2d().
  warnings.warn(
100 0.8722814321517944
Figure(600x600)
200 0.7646999359130859
Figure(600x600)
300 0.7591944932937622
Figure(600x600)
400 0.7284680008888245
Figure(600x600)
500 0.7446451187133789
Figure(600x600)
600 0.7582764029502869
Figure(600x600)
700 0.7499991655349731
Figure(600x600)
800 0.6828750967979431
Figure(600x600)
900 0.7018487453460693
Figure(600x600)
1000 0.654137074947357